In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Ti
VRAM: 12.9 GB


In [2]:
import os
os.makedirs("./flan-t5-xl", exist_ok=True)

# Small config/tokenizer files
!curl -L -o ./flan-t5-xl/config.json https://huggingface.co/google/flan-t5-xl/resolve/main/config.json
!curl -L -o ./flan-t5-xl/tokenizer.json https://huggingface.co/google/flan-t5-xl/resolve/main/tokenizer.json
!curl -L -o ./flan-t5-xl/tokenizer_config.json https://huggingface.co/google/flan-t5-xl/resolve/main/tokenizer_config.json
!curl -L -o ./flan-t5-xl/spiece.model https://huggingface.co/google/flan-t5-xl/resolve/main/spiece.model
!curl -L -o ./flan-t5-xl/generation_config.json https://huggingface.co/google/flan-t5-xl/resolve/main/generation_config.json

# Model weight shards (~5GB each)
!curl -L -o ./flan-t5-xl/model.safetensors.index.json https://huggingface.co/google/flan-t5-xl/resolve/main/model.safetensors.index.json
!curl -L -o ./flan-t5-xl/model-00001-of-00002.safetensors https://huggingface.co/google/flan-t5-xl/resolve/main/model-00001-of-00002.safetensors
!curl -L -o ./flan-t5-xl/model-00002-of-00002.safetensors https://huggingface.co/google/flan-t5-xl/resolve/main/model-00002-of-00002.safetensors

print("\nFinal files:")
!dir flan-t5-xl

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100    238 100    238   0      0   1199      0                              0
100    238 100    238   0      0   1198      0                              0

  0      0   0      0   0      0      0      0                              0
100   1438 100   1438   0      0   6066      0                              0
100   1438 100   1438   0      0   6064      0                              0
100   1438 100   1438   0      0   6062      0                              0
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100    244 100    244   0      0   1387      0               


Final files:


  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100   1379 100   1379   0      0   6954      0                              0
100   1379 100   1379   0      0   6951      0                              0

  0      0   0      0   0      0      0      0                              0
  0  1.81G   0 15.93M   0      0 13.29M      0   02:19   00:01   02:18 15.93M
  3  1.81G   3 70.27M   0      0 31.98M      0   00:58   00:02   00:56 35.15M
  6  1.81G   6 124.5M   0      0 38.97M      0   00:47   00:03   00:44 41.55M
  9  1.81G   9 173.0M   0      0 41.24M      0   00:45   00:04   00:41 43.29M
 12  1.81G  12 227.4M   0      0 43.77M      0   00:42   00:05   00:37 45.51M
 15  1.81G  15 281.8M   0      0 45.50M      0   00:40   00:06   00:34 53.21M
 18  1.81G  18 335.4M   0      0 46.64M      0   00:39   00:07

 Volume in drive C has no label.
 Volume Serial Number is B488-E100

 Directory of c:\Users\grege\OneDrive\Documents\GitHub\INFO371a\flan-t5-xl

24-Apr-26  15:34    <DIR>          .
24-Apr-26  15:31    <DIR>          ..
24-Apr-26  15:31             1,438 config.json
24-Apr-26  15:31               147 generation_config.json
24-Apr-26  15:34     9,449,619,912 model-00001-of-00002.safetensors
24-Apr-26  15:34     1,949,477,672 model-00002-of-00002.safetensors
24-Apr-26  15:31            53,032 model.safetensors.index.json
24-Apr-26  15:31           791,656 spiece.model
24-Apr-26  15:31         2,424,064 tokenizer.json
24-Apr-26  15:31             2,539 tokenizer_config.json
               8 File(s) 11,402,370,460 bytes
               2 Dir(s)  11,763,269,632 bytes free


In [3]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

import pandas as pd
import numpy as np
import torch
import re
import string
import ast
from collections import Counter
from tqdm import tqdm
from transformers import AutoTokenizer, T5ForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TOP_K = 10
MAX_INPUT_LENGTH = 1024

print("Loading Flan-T5-XL from local files...")
tokenizer = AutoTokenizer.from_pretrained("./flan-t5-xl", local_files_only=True)
model = T5ForConditionalGeneration.from_pretrained(
    "./flan-t5-xl",
    local_files_only=True,
    torch_dtype=torch.float16,
).to(device)
model.eval()
print(f"Loaded Flan-T5-XL with {sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters")

Device: cuda
Loading Flan-T5-XL from local files...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded Flan-T5-XL with 2850M parameters


In [4]:
passage_df = pd.read_parquet("data/passage_corpus.parquet")
retrieval_df = pd.read_parquet("data/dpr_retrieval_details.parquet")
gold_df = pd.read_parquet("data/gold_mapping.parquet")

passage_lookup = dict(zip(passage_df["passage_id"], passage_df["text"]))

print(f"Loaded {len(passage_df):,} passages")
print(f"Loaded retrieval results for {len(retrieval_df)} questions")
print(f"Loaded gold answers for {len(gold_df)} questions")

Loaded 95,202 passages
Loaded retrieval results for 824 questions
Loaded gold answers for 824 questions


In [5]:
def build_zero_shot_prompt(question, passage_texts):
    """Simple Context/Question/Answer template — works well with Flan-T5."""
    context = " ".join(passage_texts)
    return f"Context: {context}\nQuestion: {question}\nAnswer:"


FEW_SHOT_EXAMPLES = [
    {
        "context": ("The Eiffel Tower in Paris is 330 meters tall. "
                    "The Empire State Building in New York is 381 meters tall."),
        "question": "How many meters taller is the Empire State Building than the Eiffel Tower?",
        "answer": "51"
    },
    {
        "context": ("World War II ended in 1945. "
                    "The United Nations was founded later that same year, on October 24, 1945."),
        "question": "In what year was the United Nations founded?",
        "answer": "1945"
    },
    {
        "context": ("Barack Obama served as the 44th President of the United States from 2009 to 2017. "
                    "He was born in Honolulu, Hawaii on August 4, 1961."),
        "question": "Where was the 44th president of the United States born?",
        "answer": "Honolulu, Hawaii"
    },
]


def build_few_shot_prompt(question, passage_texts, examples=FEW_SHOT_EXAMPLES):
    parts = []
    for ex in examples:
        parts.append(f"Context: {ex['context']}")
        parts.append(f"Question: {ex['question']}")
        parts.append(f"Answer: {ex['answer']}\n")

    context = " ".join(passage_texts)
    parts.append(f"Context: {context}")
    parts.append(f"Question: {question}")
    parts.append("Answer:")
    return "\n".join(parts)

In [ ]:
def generate_answer(prompt, max_new_tokens=64):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False,
            early_stopping=True,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


test_q = "What year was the premiere of The Magic Flute?"
test_passages = ["The Magic Flute premiered on 30 September 1791 in Vienna."]
test_prompt = build_zero_shot_prompt(test_q, test_passages)
print(f"Q: {test_q}")
print(f"A: {generate_answer(test_prompt)}")

Q: What year was the premiere of The Magic Flute?
A: 1791


In [7]:
def run_generation(gold_df, retrieval_df, passage_lookup, mode="zero_shot", top_k=10):
    assert mode in ("zero_shot", "few_shot")
    build_prompt = build_zero_shot_prompt if mode == "zero_shot" else build_few_shot_prompt

    results = []
    for idx in tqdm(range(len(gold_df)), desc=f"Generating ({mode})"):
        question = gold_df.iloc[idx]["prompt"]
        gold_answer = gold_df.iloc[idx]["answer"]

        retrieved_ids = retrieval_df.iloc[idx]["retrieved_passage_ids"]
        if isinstance(retrieved_ids, str):
            retrieved_ids = ast.literal_eval(retrieved_ids)

        top_k_ids = retrieved_ids[:top_k]
        passage_texts = [passage_lookup[pid] for pid in top_k_ids if pid in passage_lookup]

        prompt = build_prompt(question, passage_texts)
        try:
            prediction = generate_answer(prompt)
        except Exception as e:
            prediction = f"[ERROR: {str(e)[:100]}]"

        gold_ids = gold_df.iloc[idx]["gold_passage_ids"]
        if isinstance(gold_ids, str):
            gold_ids = ast.literal_eval(gold_ids)
        hit_at_k = len(set(top_k_ids) & set(gold_ids)) > 0

        results.append({
            "question_idx": idx,
            "question": question,
            "gold_answer": gold_answer,
            "predicted_answer": prediction,
            "hit_at_k": hit_at_k,
            "mode": mode,
        })

    return pd.DataFrame(results)


# Run zero-shot
zs_df = run_generation(gold_df, retrieval_df, passage_lookup, mode="zero_shot", top_k=TOP_K)
zs_df.to_csv(os.path.join(OUTPUT_DIR, f"flan_t5_xl_zero_shot_k{TOP_K}.csv"), index=False)
print(f"\nSample zero-shot predictions:")
print(zs_df[["question", "gold_answer", "predicted_answer"]].head(5).to_string())

Generating (zero_shot): 100%|██████████| 824/824 [02:44<00:00,  5.01it/s]


Sample zero-shot predictions:
                                                                                                                                                                                                                                                                            question  gold_answer predicted_answer
0                                                             If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated president's mother's maiden name, what is my future wife's name?   Jane Ballou             Mary
1  Imagine there is a building called Bronte tower whose height in feet is the same number as the dewey decimal classification for the Charlotte Bronte book that was published in 1847. Where would this building rank among tallest buildings in New York City, as of August 2024?         37th              5th
2                                               

In [8]:
fs_df = run_generation(gold_df, retrieval_df, passage_lookup, mode="few_shot", top_k=TOP_K)
fs_df.to_csv(os.path.join(OUTPUT_DIR, f"flan_t5_xl_few_shot_k{TOP_K}.csv"), index=False)
print(f"\nSample few-shot predictions:")
print(fs_df[["question", "gold_answer", "predicted_answer"]].head(5).to_string())

Generating (few_shot): 100%|██████████| 824/824 [02:45<00:00,  4.98it/s]



Sample few-shot predictions:
                                                                                                                                                                                                                                                                            question  gold_answer predicted_answer
0                                                             If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated president's mother's maiden name, what is my future wife's name?   Jane Ballou     unanswerable
1  Imagine there is a building called Bronte tower whose height in feet is the same number as the dewey decimal classification for the Charlotte Bronte book that was published in 1847. Where would this building rank among tallest buildings in New York City, as of August 2024?         37th              5th
2                                                

In [10]:
def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        return "".join(ch for ch in text if ch not in set(string.punctuation))
    return white_space_fix(remove_articles(remove_punc(str(s).lower())))


def exact_match(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))


def token_f1(pred, gold):
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()

    if not pred_tokens or not gold_tokens:
        return int(pred_tokens == gold_tokens)

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def evaluate(df, label):
    df = df.copy()
    df["em"] = df.apply(lambda r: exact_match(r["predicted_answer"], r["gold_answer"]), axis=1)
    df["f1"] = df.apply(lambda r: token_f1(r["predicted_answer"], r["gold_answer"]), axis=1)

    overall_em = df["em"].mean()
    overall_f1 = df["f1"].mean()
    hit = df[df["hit_at_k"]]
    miss = df[~df["hit_at_k"]]

    print(f"\n{'=' * 55}")
    print(f"Evaluation — {label}")
    print(f"{'=' * 55}")
    print(f"Overall:              EM = {overall_em:.4f}  F1 = {overall_f1:.4f}  (n={len(df)})")
    print(f"Retrieval hit (k={TOP_K}):   EM = {hit['em'].mean():.4f}  F1 = {hit['f1'].mean():.4f}  (n={len(hit)})")
    print(f"Retrieval miss (k={TOP_K}):  EM = {miss['em'].mean():.4f}  F1 = {miss['f1'].mean():.4f}  (n={len(miss)})")
    print(f"{'=' * 55}")

    return df, {
        "label": label,
        "em": overall_em,
        "f1": overall_f1,
        "em_hit": hit["em"].mean() if len(hit) else None,
        "em_miss": miss["em"].mean() if len(miss) else None,
        "f1_hit": hit["f1"].mean() if len(hit) else None,
        "f1_miss": miss["f1"].mean() if len(miss) else None,
    }


zs_df, zs_metrics = evaluate(zs_df, "Zero-shot (XL, k=10)")
fs_df, fs_metrics = evaluate(fs_df, "Few-shot (XL, k=10)")

zs_df.to_csv(os.path.join(OUTPUT_DIR, f"flan_t5_xl_zero_shot_evaluated_k{TOP_K}.csv"), index=False)
fs_df.to_csv(os.path.join(OUTPUT_DIR, f"flan_t5_xl_few_shot_evaluated_k{TOP_K}.csv"), index=False)

summary = pd.DataFrame([zs_metrics, fs_metrics])
summary.to_csv(os.path.join(OUTPUT_DIR, f"flan_t5_xl_summary_k{TOP_K}.csv"), index=False)
print("\nFinal summary metrics:")
print(summary.to_string(index=False))


Evaluation — Zero-shot (XL, k=10)
Overall:              EM = 0.0328  F1 = 0.0601  (n=824)
Retrieval hit (k=10):   EM = 0.0000  F1 = 0.1556  (n=9)
Retrieval miss (k=10):  EM = 0.0331  F1 = 0.0591  (n=815)

Evaluation — Few-shot (XL, k=10)
Overall:              EM = 0.0231  F1 = 0.0357  (n=824)
Retrieval hit (k=10):   EM = 0.0000  F1 = 0.0000  (n=9)
Retrieval miss (k=10):  EM = 0.0233  F1 = 0.0361  (n=815)

Final summary metrics:
               label       em       f1  em_hit  em_miss   f1_hit  f1_miss
Zero-shot (XL, k=10) 0.032767 0.060111     0.0 0.033129 0.155556 0.059057
 Few-shot (XL, k=10) 0.023058 0.035664     0.0 0.023313 0.000000 0.036058


In [ ]:

print("retrieval_df columns:", list(retrieval_df.columns))
print("retrieval_df length:", len(retrieval_df))
print("\nFirst row of retrieval_df:")
print(retrieval_df.iloc[0])

print("\nType of retrieved_passage_ids:", type(retrieval_df.iloc[0]["retrieved_passage_ids"]))
print("First 5 retrieved IDs:", retrieval_df.iloc[0]["retrieved_passage_ids"][:5] if hasattr(retrieval_df.iloc[0]["retrieved_passage_ids"], "__getitem__") else "not indexable")

print("\nFirst row of gold_df gold_passage_ids:")
print(gold_df.iloc[0]["gold_passage_ids"])
print("Type:", type(gold_df.iloc[0]["gold_passage_ids"]))

if "hit_at_10" in retrieval_df.columns:
    print(f"\nRetrieval hits @10 from Step 2: {retrieval_df['hit_at_10'].sum()} / {len(retrieval_df)}")

retrieval_df columns: ['question_idx', 'retrieved_passage_ids', 'gold_passage_ids', 'hit_at_10']
retrieval_df length: 824

First row of retrieval_df:
question_idx                                                             0
retrieved_passage_ids    [58505, 36758, 17493, 60960, 57912, 25799, 239...
gold_passage_ids         [8574, 8575, 8576, 20742, 20743, 20744, 20745,...
hit_at_10                                                            False
Name: 0, dtype: object

Type of retrieved_passage_ids: <class 'numpy.ndarray'>
First 5 retrieved IDs: [58505 36758 17493 60960 57912]

First row of gold_df gold_passage_ids:
[ 8505  8506  8507  8508  8509  8510  8511  8512  8513  8514  8515  8516
  8517  8518  8519  8520  8521  8522  8523  8524  8525  8526  8527  8528
  8529  8530  8531  8532  8533  8534  8535  8536  8537  8538  8539  8540
  8541  8542  8543  8544  8545  8546  8547  8548  8549  8550  8551  8552
  8553  8554  8555  8556  8557  8558  8559  8560  8561  8562  8563  8564
  8565  856

In [ ]:

gold_lookup = {}
for _, row in gold_df.iterrows():
    gids = row["gold_passage_ids"]
    if isinstance(gids, str):
        gids = ast.literal_eval(gids)
    gold_lookup[int(row["question_idx"])] = set(int(x) for x in gids)

print(f"Built gold_lookup for {len(gold_lookup)} questions")

# Recompute hit_at_k for both zero-shot and few-shot results
def recompute_hit(df, retrieval_df, gold_lookup, k=10):
    df = df.copy()
    retrieval_lookup = dict(zip(retrieval_df["question_idx"], retrieval_df["retrieved_passage_ids"]))

    new_hits = []
    for idx in df["question_idx"]:
        retrieved = retrieval_lookup.get(int(idx), [])
        top_k = set(int(x) for x in list(retrieved)[:k])
        gold = gold_lookup.get(int(idx), set())
        new_hits.append(len(top_k & gold) > 0)

    df["hit_at_k"] = new_hits
    return df

zs_df = recompute_hit(zs_df, retrieval_df, gold_lookup, k=TOP_K)
fs_df = recompute_hit(fs_df, retrieval_df, gold_lookup, k=TOP_K)

print(f"\nZero-shot hits @{TOP_K}: {zs_df['hit_at_k'].sum()} / {len(zs_df)}")
print(f"Few-shot hits  @{TOP_K}: {fs_df['hit_at_k'].sum()} / {len(fs_df)}")

zs_df, zs_metrics = evaluate(zs_df, "Zero-shot (XL, k=10)")
fs_df, fs_metrics = evaluate(fs_df, "Few-shot (XL, k=10)")

zs_df.to_csv(os.path.join(OUTPUT_DIR, f"flan_t5_xl_zero_shot_evaluated_k{TOP_K}.csv"), index=False)
fs_df.to_csv(os.path.join(OUTPUT_DIR, f"flan_t5_xl_few_shot_evaluated_k{TOP_K}.csv"), index=False)

summary = pd.DataFrame([zs_metrics, fs_metrics])
summary.to_csv(os.path.join(OUTPUT_DIR, f"flan_t5_xl_summary_k{TOP_K}.csv"), index=False)
print("\nFinalsummary:")
print(summary.to_string(index=False))

Built gold_lookup for 824 questions

Zero-shot hits @10: 9 / 824
Few-shot hits  @10: 9 / 824

Evaluation — Zero-shot (XL, k=10)
Overall:              EM = 0.0328  F1 = 0.0601  (n=824)
Retrieval hit (k=10):   EM = 0.0000  F1 = 0.1556  (n=9)
Retrieval miss (k=10):  EM = 0.0331  F1 = 0.0591  (n=815)

Evaluation — Few-shot (XL, k=10)
Overall:              EM = 0.0231  F1 = 0.0357  (n=824)
Retrieval hit (k=10):   EM = 0.0000  F1 = 0.0000  (n=9)
Retrieval miss (k=10):  EM = 0.0233  F1 = 0.0361  (n=815)

Final (corrected) summary:
               label       em       f1  em_hit  em_miss   f1_hit  f1_miss
Zero-shot (XL, k=10) 0.032767 0.060111     0.0 0.033129 0.155556 0.059057
 Few-shot (XL, k=10) 0.023058 0.035664     0.0 0.023313 0.000000 0.036058


In [13]:
# Manually verify retrieval quality on the first question
print("=" * 60)
print("Question 0 — diagnostic")
print("=" * 60)

# Get first row from retrieval_df and gold_df
ret_row = retrieval_df.iloc[0]
gold_row = gold_df.iloc[0]

print(f"\nretrieval_df question_idx: {ret_row['question_idx']}")
print(f"gold_df question_idx:      {gold_row['question_idx']}")
print(f"\ngold_df prompt: {gold_row['prompt'][:150]}")

retrieved = list(ret_row["retrieved_passage_ids"])
gold = list(gold_row["gold_passage_ids"])

print(f"\nRetrieved top-10: {retrieved[:10]}")
print(f"Gold passages (first 10): {gold[:10]}")
print(f"Number of gold passages: {len(gold)}")
print(f"Overlap: {set(int(x) for x in retrieved[:10]) & set(int(x) for x in gold)}")

# Check if retrieved IDs are even in the passage_df
print(f"\nMax passage_id in passage_df: {passage_df['passage_id'].max()}")
print(f"Max retrieved ID here:         {max(retrieved)}")
print(f"Max gold ID here:              {max(gold)}")

# Sample passage texts
print(f"\nSample retrieved passage (ID {retrieved[0]}):")
p = passage_df[passage_df["passage_id"] == retrieved[0]]
if len(p): print(p.iloc[0]["text"][:200])

print(f"\nSample gold passage (ID {gold[0]}):")
p = passage_df[passage_df["passage_id"] == gold[0]]
if len(p): print(p.iloc[0]["text"][:200])

Question 0 — diagnostic

retrieval_df question_idx: 0
gold_df question_idx:      0

gold_df prompt: If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated p

Retrieved top-10: [np.int64(58505), np.int64(36758), np.int64(17493), np.int64(60960), np.int64(57912), np.int64(25799), np.int64(23945), np.int64(47312), np.int64(94313), np.int64(83513)]
Gold passages (first 10): [np.int64(8505), np.int64(8506), np.int64(8507), np.int64(8508), np.int64(8509), np.int64(8510), np.int64(8511), np.int64(8512), np.int64(8513), np.int64(8514)]
Number of gold passages: 394
Overlap: set()

Max passage_id in passage_df: 95201
Max retrieved ID here:         94313
Max gold ID here:              94688

Sample retrieved passage (ID 58505):
William II (or in Occitan: Guilhem II) was the second Lord of Montpellier.

Sample gold passage (ID 8505):
James Abram Garfield (November 19, 1831 – September 19, 1881) was the 20

In [ ]:
# Check if retrieval IDs align with the passage_df
print("Does retrieved ID 58505 exist in passage_df?", 
      (passage_df["passage_id"] == 58505).any())
print("Does gold ID 8505 exist in passage_df?", 
      (passage_df["passage_id"] == 8505).any())
print("passage_df shape:", passage_df.shape)
print("passage_df passage_id range:", passage_df["passage_id"].min(), "to", passage_df["passage_id"].max())

# Look at the gold_df - how many gold passages per question on average?
gold_df["n_gold_from_ids"] = gold_df["gold_passage_ids"].apply(lambda x: len(x))
print(f"\nGold passages per question: min={gold_df['n_gold_from_ids'].min()}, median={gold_df['n_gold_from_ids'].median()}, mean={gold_df['n_gold_from_ids'].mean():.1f}, max={gold_df['n_gold_from_ids'].max()}")

hits_at_10 = 0
for idx in range(len(retrieval_df)):
    retrieved_top10 = set(int(x) for x in list(retrieval_df.iloc[idx]["retrieved_passage_ids"])[:10])
    gold_ids = set(int(x) for x in list(gold_df.iloc[idx]["gold_passage_ids"]))
    if retrieved_top10 & gold_ids:
        hits_at_10 += 1

print(f"\nFresh Recall@10 computation: {hits_at_10}/{len(retrieval_df)} = {hits_at_10/len(retrieval_df):.4f}")

Does retrieved ID 58505 exist in passage_df? True
Does gold ID 8505 exist in passage_df? True
passage_df shape: (95202, 6)
passage_df passage_id range: 0 to 95201

Gold passages per question: min=2, median=105.0, mean=126.4, max=898

Fresh Recall@10 computation: 9/824 = 0.0109


In [15]:
import os
for f in ["data/passage_index.faiss", "data/passage_embeddings.npy", "data/passage_corpus.parquet"]:
    exists = os.path.exists(f)
    size = os.path.getsize(f) / 1e6 if exists else 0
    print(f"{'✓' if exists else '✗'} {f}: {size:.1f} MB")

✗ data/passage_index.faiss: 0.0 MB
✗ data/passage_embeddings.npy: 0.0 MB
✓ data/passage_corpus.parquet: 35.1 MB
